# Fixing QPSK — composite-window recall

## The problem, measured

> **These numbers came from the 2026-09-22 `radar-fix-data` build**, which has since
> been set aside. The mechanism below is structural and should carry over, but the
> exact figures will shift on a different dataset — **section 3 re-measures the
> baseline on whatever data you actually load**, and that is the number to judge
> against, not this table.

On the held-out test split at SNR >= +2 dB, with the shipped `best_model.pt`:

| class | standalone | composite | blended |
|---|---|---|---|
| BPSK | 1.000 | 0.557 | 0.874 |
| **QPSK** | **0.997** | **0.061** | **0.730** |
| 16QAM | 0.941 | 0.559 | 0.832 |
| 64QAM | 0.942 | 0.415 | 0.821 |

QPSK is recognised **essentially perfectly on its own** and almost never when a
radar, hopper or jammer shares the window. This is not a modulation-recognition
problem and not an SNR problem.

## What is actually going wrong

On composite QPSK windows the model's scores are:

| class | standalone | composite |
|---|---|---|
| QPSK | 0.979 | **0.180** |
| 16QAM | 0.006 | **0.218** |
| 64QAM | 0.004 | **0.199** |

16QAM and 64QAM *rise above* QPSK. The evidence is not lost — the model still
sees a linearly-modulated civilian signal, it just **assigns the wrong
constellation order**.

Why QPSK specifically: it is four tight clusters. A co-present emitter smears
them into a diffuse cloud, and a diffuse cloud is what a denser constellation
looks like. 16QAM survives because it is already a cloud. BPSK survives because
two points is the most distinctive shape there is. QPSK sits in the worst spot.

## The target

Standalone is already 0.997 and composites are 28.6% of QPSK windows, so:

    0.714 x 0.997  +  0.286 x C  >=  0.80   ->   C >= 0.307

**Composite QPSK recall has to reach ~0.31.** It is 0.061 now. For scale, BPSK
already manages 0.557 and 16QAM 0.559 on the same composite load — so the target
is *below* what sibling classes already achieve.

## The two levers

**A — `cumulant_features`** (Eileen's branch, cherry-picked here). Adds |C40|,
|C42|, |C63| after an RRC matched filter as expert features. Cumulants above
order 2 are mathematically **zero for Gaussian noise**, so they read the
constellation *through* smearing. QPSK's |C42| is 1.0 against 16QAM's 0.680 —
a gap ~5x larger than the 16QAM/64QAM case this was originally built for.

*Honest caveat:* that zero-for-Gaussian property holds for **barrage jamming**
(band-limited Gaussian noise), which is exactly the worst case today (0.007).
Radar chirps and FHSS tones are constant-modulus and have their own non-zero
cumulants, so expect less help there.

**B — `composite_pos_weight`.** Today `pos_weight` is per-*class*, identical for
every window, so missing QPSK costs the same whether it sits alone or under a
jammer — and 71% of QPSK windows are standalone, so the gradient says "describe
whatever dominates." This makes multi-label windows worth N times as much in the
training loss. Validation loss stays unweighted, so "best epoch" still means the
same thing and val losses stay comparable across cells.

A attacks the *confusion*; B attacks the *incentive*. They are independent, so
they get their own cells plus one combined.

## 1. Code

Must be the `qpsk-cumulant-features` branch — `main` does not have the cumulant code.

In [ ]:
BRANCH = 'qpsk-cumulant-features'

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b $BRANCH https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -2

In [ ]:
!pip install -q pyyaml h5py

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY - stop and switch to a GPU runtime (Runtime > Change runtime type)')

## 2. Data

From `MyDrive/sedic/eavan-retrain/`. Change `DRIVE_DATA` if the folder moves — the
rest of the notebook re-measures everything against whatever is loaded here, so it
stays correct on any dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/sedic/eavan-retrain'
!mkdir -p data/processed
!cp $DRIVE_DATA/X.npy data/processed/
!cp $DRIVE_DATA/y.npy data/processed/
!cp $DRIVE_DATA/snr_labels.npy data/processed/
!ls -la data/processed/

In [ ]:
import numpy as np
from src.config import CFG, CLASSES

X = np.load('data/processed/X.npy', mmap_mode='r')
y = np.load('data/processed/y.npy')
snr = np.load('data/processed/snr_labels.npy')
assert X.ndim == 3 and X.shape[1] == 2, X.shape
assert y.shape[1] == len(CLASSES), y.shape
assert sorted(set(snr.tolist())) == [float(b) for b in sorted(CFG['snr_bins_db'])]
print('X', X.shape, '| y', y.shape, '| SNR bins OK')

q = CLASSES.index('QPSK')
pos = y[:, q] == 1
print(f"QPSK windows {pos.sum():,} -- standalone {(pos & (y.sum(1)==1)).sum():,}, "
      f"composite {(pos & (y.sum(1)>1)).sum():,}")

### Confirm the flags apply

Seconds, and it catches a stale clone before hours of GPU time. `fc1.in` must grow
from 192 to **195** when `cumulant_features` is on — that is the three cumulants.

In [ ]:
from src.models.amc_cnn import AMC_CNN
L = CFG['signal']['window_len']
for name, kw in [('baseline',  dict(stft_freq_summary=False, stft_keep_rows=False, cumulant_features=False)),
                 ('cumulant',  dict(stft_freq_summary=False, stft_keep_rows=False, cumulant_features=True))]:
    m = AMC_CNN(num_classes=len(CLASSES), input_len=L, **kw)
    assert (m.cumulant_branch is not None) == kw['cumulant_features']
    print(f"{name:<10} params {sum(p.numel() for p in m.parameters()):>8,}   fc1.in {m.fc1.in_features}")
    del m

## 3. Baseline — measure before changing anything

**Do not skip this.** Every earlier number came from a different model or a
different dataset, so none of them are comparable. This cell produces the only
reference the three experiments should be judged against, and it also re-derives
the composite-recall target for *this* data (the report prints it at the bottom —
"composite recall must reach X").

In [ ]:
OUT    = '/content/drive/MyDrive/sedic/qpsk'
EPOCHS = 15      # baseline converged around epoch 10 last time; 15 leaves margin
!mkdir -p $OUT
print(OUT, '| epochs', EPOCHS)

In [ ]:
!python scripts/run_stft_experiment.py --no-freq-summary --epochs $EPOCHS --out-dir $OUT

In [ ]:
!python scripts/civilian_composite_report.py --checkpoint $OUT/experiment_baseline.pt

## 4. The three experiments

Each writes its own checkpoint to Drive, so a disconnect costs at most the cell in
flight, and you can run them across separate sessions.

| cell | tag | lever |
|---|---|---|
| A | `cum` | cumulant features — fixes the *confusion* |
| B | `cw4` | composite loss weight 4x — fixes the *incentive* |
| A+B | `cum_cw4` | both |

In [ ]:
!python scripts/run_stft_experiment.py --no-freq-summary --cumulant-features --epochs $EPOCHS --out-dir $OUT

In [ ]:
!python scripts/run_stft_experiment.py --no-freq-summary --composite-weight 4 --epochs $EPOCHS --out-dir $OUT

In [ ]:
!python scripts/run_stft_experiment.py --no-freq-summary --cumulant-features --composite-weight 4 --epochs $EPOCHS --out-dir $OUT

## 5. Score them

The report reads each checkpoint's own `_config.json` sidecar, so the flags can
never drift out of sync with the weights.

**Read the `composite` column.** That is the one that has to move from 0.061 toward
0.31. `standalone` should stay near 1.0 — if it drops, the change cost you
something and the blended number may not have improved at all.

Also watch the per-class score block at the bottom: if the fix is working for the
right reason, **QPSK's mean score should rise above 16QAM's and 64QAM's** on
composite QPSK windows.

In [ ]:
import pathlib
for tag in ['baseline', 'cum', 'cw4', 'cum_cw4']:
    p = pathlib.Path(OUT) / f'experiment_{tag}.pt'
    if not p.exists():
        print(f'--- {tag}: not run yet ---\n'); continue
    print('=' * 70); print(tag.upper()); print('=' * 70)
    !python scripts/civilian_composite_report.py --checkpoint $p
    print()

## 6. Check you did not break the judged classes

QPSK is **not** a judged class — only LFM_RADAR, FHSS and JAMMING are scored
against the 80% bar (`judged_classes` in the config). So a QPSK gain that costs
any of those three is a bad trade, and this is where that shows up.

In [ ]:
import numpy as np, torch, json, pathlib
from sklearn.metrics import average_precision_score
from src.config import CFG, CLASSES, CLASS_TO_IDX
from src.models.amc_cnn import AMC_CNN
from src.train import load_data, stratified_split

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X, y, snr = load_data(); d = CFG['dataset']
_, _, te = stratified_split(y, snr, d['val_frac'], d['test_frac'], d['seed'])
Xte, yte = X[te], y[te].astype(int)

print(f"{'run':<12} | " + ' '.join(f'{c:>11}' for c in ['LFM_RADAR','FHSS','JAMMING']) + ' |    mean')
print('-' * 62)
for tag in ['baseline', 'cum', 'cw4', 'cum_cw4']:
    ck = pathlib.Path(OUT) / f'experiment_{tag}.pt'
    if not ck.exists(): continue
    f = json.loads((ck.parent / f'{ck.stem}_config.json').read_text())
    m = AMC_CNN(num_classes=len(CLASSES), input_len=CFG['signal']['window_len'],
                stft_freq_summary=f.get('stft_freq_summary', False),
                stft_keep_rows=f.get('stft_keep_rows', False),
                cumulant_features=f.get('cumulant_features', False)).to(dev)
    m.load_state_dict(torch.load(ck, map_location=dev)); m.eval()
    sc = []
    with torch.no_grad():
        for i in range(0, len(Xte), 512):
            sc.append(torch.sigmoid(m(torch.tensor(Xte[i:i+512], dtype=torch.float32).to(dev))).cpu().numpy())
    sc = np.concatenate(sc); del m
    aps = [average_precision_score(yte[:, CLASS_TO_IDX[c]], sc[:, CLASS_TO_IDX[c]])
           for c in ['LFM_RADAR','FHSS','JAMMING']]
    print(f"{tag:<12} | " + ' '.join(f'{a:>11.4f}' for a in aps) + f' | {np.mean(aps):>7.4f}')

## 7. Reading the result

- **Composite QPSK >= 0.31 and judged-class AP not down** — it worked. Pick that
  cell, retrain as the 5-seed ensemble, recalibrate, re-evaluate.
- **Composite QPSK moved but under 0.31** — the direction is right. Try
  `--composite-weight 8`, or A+B if you only ran them separately.
- **`cum` alone moved it, `cw4` alone did not** — it was a confusion problem, as
  diagnosed. Keep cumulants, drop the loss change.
- **`cw4` alone moved it, `cum` did not** — it was an incentive problem. The
  representation could carry both emitters all along, it was just never asked to.
- **Neither moved it** — the fused representation cannot carry two emitters at
  once, and the real fix is per-class attention pooling (one `AttentionPool1d` is
  currently shared by all 8 outputs, so it can only look in one place at a time).
  That is a real architecture change, not a flag.

Nothing needs rescuing at the end — everything already lives in `sedic/qpsk/` on
Drive.

**Do not set these flags to `true` in `configs/default.yaml` yet.** With a flag on,
the model's shape changes and none of the shipped `results/*.pt` will load —
`src/evaluate.py`, `calibrate_thresholds.py`, the UI console and the static site
build all construct a model from that config. Flip the config only in the same step
as installing matching checkpoints.